### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [3]:
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(raw_docs)

In [4]:
chunks[0:5]

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [5]:
len(chunks)

241

In [6]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k': 5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000028D108EDFD0>, search_type='mmr', search_kwargs={'k': 5})

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

llm = init_chat_model(
    model="openai:gpt-4.1-mini",
    api_key=os.getenv("GPT-4-MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)
llm

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028D3EBE40B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000028D5C60C950>, root_client=<openai.OpenAI object at 0x0000028D39805AF0>, root_async_client=<openai.AsyncOpenAI object at 0x0000028D5C60C980>, model_name='gpt-4.1-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://models.inference.ai.azure.com')

In [9]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000028D3EBE40B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000028D5C60C950>, root_client=<openai.OpenAI object at 0x

In [10]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)

In [11]:
# Step 5 : RAG pipeline
rag_pipeline = (
    RunnableMap({
        "input": lambda x : x["input"],
        "context": lambda x : retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    }
    )
    | document_chain
)

In [12]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer:\n", response)

Expanded query:  
{"input": "What types of memory modules and storage mechanisms does LangChain support? Please include details on supported memory interfaces such as short-term memory, long-term memory, conversation memory, vector stores, knowledge bases, session persistence, caching strategies, and any integrations with databases or external memory systems for maintaining state or context in language model applications."}
Answer:
 LangChain supports memory types such as **ConversationBufferMemory** and **ConversationSummaryMemory**. These memory modules enable the large language model (LLM) to maintain context by keeping track of previous conversation turns or by summarizing long interactions to stay within token limits.


In [13]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer:\n", response)

Expanded query:  
{'input': 'CrewAI agents OR CrewAI assistants OR autonomous CrewAI systems OR AI-powered Crew agents OR artificial intelligence crew members OR intelligent virtual crew agents OR AI-driven crew management tools OR automated crew coordination systems'}
Answer:
 CrewAI agents are autonomous agents that operate within structured workflows by forming a collaborative crew. Each agent has a defined role—such as researcher, planner, or executor—and functions semi-independently while working towards the overall crew objective. They are designed with a specific purpose, goal, and a set of tools, ensuring that each agent stays focused on its tasks and contributes meaningfully to the team's collective goals.
